## Inference under FHE for the MNIST Dataset using helayers

In this demo, we'll deal with a classification problem for the MNIST dataset [1], trying to correctly classify a batch of samples using a neural network model that will be created and trained using the Keras library (with architecture similar to reference [2]).
First, we'll build a plain neural network for the MNIST model. Then, we'll encrypt the trained network and run inference over it using FHE.

In [1]:
import os

##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)
from tensorflow.keras import backend as K

from tensorflow.keras import utils, losses
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
import h5py

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)

# batch_size = 16
# batch_size = 32
# batch_size = 64
# batch_size = 128
# batch_size = 256
# batch_size = 512
batch_size = 1024
# batch_size = 2048
# batch_size = 4096
# batch_size=8192
print("Misc. initializations")

2025-06-11 04:45:12.144992: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Misc. initializations


### Load and Preprocess the MNIST Dataset. 

In [2]:
# 修改导入模块（原MNIST模块替换为Fashion-MNIST模块）
from tensorflow.keras.datasets import fashion_mnist

# 加载Fashion-MNIST数据集（原mnist.load_data()替换为fashion_mnist.load_data()）
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# 以下预处理步骤与MNIST完全兼容，无需修改
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# 裁剪部分（根据需求选择性取消注释）
# x_train = x_train[:, 2:26, 2:26]
# x_test = x_test[:, 2:26, 2:26]

# 添加通道维度（Fashion-MNIST为灰度图，需扩展为[28,28,1]）
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# 归一化至[0,1]范围
x_train /= 255
x_test /= 255

print('Fashion-MNIST data ready')

Fashion-MNIST data ready


In [3]:
# Create validation data
# valsize=16
# valsize=500
valsize=5000
x_train = x_train[:-valsize]
x_val = x_train[-valsize:]
y_train = y_train[:-valsize]
y_val = y_train[-valsize:]
print('Validation and test data ready')

# Convert class vector to binary class matrices
# num_classes = 10
# y_train = utils.to_categorical(y_train, num_classes)
# y_test = utils.to_categorical(y_test, num_classes)
# y_val = utils.to_categorical(y_val, num_classes)

input_shape = x_train[0].shape
print(f'input shape: {input_shape}')

Validation and test data ready
input shape: (28, 28, 1)


### Save Dataset

In [4]:
def save_data_set(x, y, data_type, s=''):
    fname=os.path.join(PATH, f'x_{data_type}{s}.h5')
    print("Saving x_{} of shape {} in {}".format(data_type, x.shape,fname))
    xf = h5py.File(fname, 'w')
    xf.create_dataset('x_{}'.format(data_type), data=x)
    xf.close()

    yf = h5py.File(os.path.join(PATH, f'y_{data_type}{s}.h5'), 'w')
    yf.create_dataset(f'y_{data_type}', data=y)
    yf.close()

save_data_set(x_test, y_test, data_type='test5')
# save_data_set(x_train, y_train, data_type='train')
# save_data_set(x_val, y_val, data_type='val')

Saving x_test5 of shape (10000, 28, 28, 1) in data/net_mnist/x_test5.h5


### Build a Plain Neural Network for the MNIST Model

In [5]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D,AveragePooling2D,Softmax
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2

# 创建模型
model = Sequential()

# 保持前四层不变
model.add(Conv2D(20, kernel_size=5, kernel_regularizer=l2(0.0005),input_shape=(28, 28, 1)))
model.add(AvgPool2D(pool_size=2))

model.add(Conv2D(50, kernel_size=5, kernel_regularizer=l2(0.0005)))
model.add(AvgPool2D(pool_size=2))  # 输出 (4,4,50)

# 关键修改：直接输出10通道
model.add(Conv2D(10, kernel_size=(4,4), kernel_regularizer=l2(0.0005)))  # 输出形状 (1,1,10)
model.add(BatchNormalization())
model.add(ReLU())

# 替换全局平均池化为普通平均池化+展平
model.add(AvgPool2D(pool_size=(1,1)))  # 保持空间维度 (1x1)
model.add(Flatten())                   # 输出形状 (10,)
# 输出模型架构


model.summary()

2025-06-11 04:45:15.364314: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-11 04:45:15.706835: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-11 04:45:15.709537: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-06-11 04:45:15.713382: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the approp

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 12, 12, 20)       0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 50)          25050     
                                                                 
 average_pooling2d_1 (Averag  (None, 4, 4, 50)         0         
 ePooling2D)                                                     
                                                                 
 conv2d_2 (Conv2D)           (None, 1, 1, 10)          8010      
                                                                 
 batch_normalization (BatchN  (None, 1, 1, 10)         4

### Train the Neural Network

In [6]:
#一阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_addons as tfa

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=10,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
# def lr_scheduler(epoch):
#     if epoch < 100:
#         return 0.001
#     elif epoch < 200:
#         return 0.0005
#     else:
#         return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

model.fit(
    x_train, y_train, batch_size=batch_size,
    
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.9.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make su

Epoch 1/4000


2025-06-11 04:45:21.562361: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8500


54/54 - 6s - loss: 1.6120 - accuracy: 0.5394 - val_loss: 2.1773 - val_accuracy: 0.6432 - 6s/epoch - 113ms/step
Epoch 2/4000
54/54 - 0s - loss: 1.2424 - accuracy: 0.6979 - val_loss: 2.0691 - val_accuracy: 0.6980 - 484ms/epoch - 9ms/step
Epoch 3/4000
54/54 - 0s - loss: 1.1627 - accuracy: 0.7291 - val_loss: 1.9542 - val_accuracy: 0.7008 - 481ms/epoch - 9ms/step
Epoch 4/4000
54/54 - 0s - loss: 1.1176 - accuracy: 0.7509 - val_loss: 1.8140 - val_accuracy: 0.7262 - 484ms/epoch - 9ms/step
Epoch 5/4000
54/54 - 0s - loss: 1.0866 - accuracy: 0.7647 - val_loss: 1.6560 - val_accuracy: 0.7446 - 484ms/epoch - 9ms/step
Epoch 6/4000
54/54 - 1s - loss: 1.0636 - accuracy: 0.7730 - val_loss: 1.4894 - val_accuracy: 0.7604 - 596ms/epoch - 11ms/step
Epoch 7/4000
54/54 - 0s - loss: 1.0440 - accuracy: 0.7809 - val_loss: 1.3570 - val_accuracy: 0.7622 - 487ms/epoch - 9ms/step
Epoch 8/4000
54/54 - 0s - loss: 1.0284 - accuracy: 0.7859 - val_loss: 1.2211 - val_accuracy: 0.7752 - 484ms/epoch - 9ms/step
Epoch 9/4000


### Rebuild Model

In [7]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
new_model = Sequential()
for layer in model.layers:
    
    if isinstance(layer, ReLU):
        new_model.add(SquareActivation())
    else:
        new_layer = layer.__class__.from_config(layer.get_config())
        new_model.add(new_layer)
        if layer.weights:
            new_layer.set_weights(layer.get_weights())  # 自动复制权重和偏置

new_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 12, 12, 20)       0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 50)          25050     
                                                                 
 average_pooling2d_1 (Averag  (None, 4, 4, 50)         0         
 ePooling2D)                                                     
                                                                 
 conv2d_2 (Conv2D)           (None, 1, 1, 10)          8010      
                                                                 
 batch_normalization (BatchN  (None, 1, 1, 10)        

In [8]:
#二阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=10,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
# def lr_scheduler(epoch):
#     if epoch < 100:
#         return 0.001
#     elif epoch < 200:
#         return 0.0005
#     else:
#         return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

new_model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

new_model.fit(
    x_train, y_train, batch_size=batch_size,
   
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = new_model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000
54/54 - 1s - loss: 0.4754 - accuracy: 0.8543 - val_loss: 0.5076 - val_accuracy: 0.8348 - 1s/epoch - 23ms/step
Epoch 2/4000
54/54 - 0s - loss: 0.4549 - accuracy: 0.8560 - val_loss: 0.4781 - val_accuracy: 0.8464 - 498ms/epoch - 9ms/step
Epoch 3/4000
54/54 - 0s - loss: 0.4499 - accuracy: 0.8560 - val_loss: 0.4690 - val_accuracy: 0.8492 - 498ms/epoch - 9ms/step
Epoch 4/4000
54/54 - 1s - loss: 0.4491 - accuracy: 0.8566 - val_loss: 0.4635 - val_accuracy: 0.8522 - 501ms/epoch - 9ms/step
Epoch 5/4000
54/54 - 0s - loss: 0.4472 - accuracy: 0.8571 - val_loss: 0.4656 - val_accuracy: 0.8500 - 497ms/epoch - 9ms/step
Epoch 6/4000
54/54 - 0s - loss: 0.4466 - accuracy: 0.8562 - val_loss: 0.4599 - val_accuracy: 0.8542 - 499ms/epoch - 9ms/step
Epoch 7/4000
54/54 - 0s - loss: 0.4455 - accuracy: 0.8567 - val_loss: 0.4600 - val_accuracy: 0.8546 - 496ms/epoch - 9ms/step
Epoch 8/4000
54/54 - 0s - loss: 0.4453 - accuracy: 0.8575 - val_loss: 0.4540 - val_accuracy: 0.8560 - 496ms/epoch - 9ms/step
Ep

### Serialize Model and Weights

In [9]:
model_json = new_model.to_json()
with open(os.path.join(PATH, 'model608.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
new_model.save_weights(os.path.join(PATH, 'model608.h5'))
print("Saved model to disk")

Saved model to disk


We are all done training the plain network. Next we will encrypt the network and run inference over it using FHE. Let's start with some initializations.

In [ ]:
import pyhelayers
import utils

utils.verify_memory()

print('Misc. initializations')

In [ ]:
import pyhelayers
import utils
import h5py
import os
import numpy as np
##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
# from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)



utils.verify_memory()

print('Misc. initializations')

batch_size=500

The following is a general outline of the next steps.

We encode and encrypt a neural network model, using the files that were created and saved before. An automated optimizer, that occurs during the call to encode_encrypt, will examine our network and will determine various configuration details that will allow running inference under encryption efficiently.

Next, we will demonstrate how we can encrypt data, run inference on our encrypted network, and compare the results against the expected labels.
Now let's dive in . . .

In [ ]:
he_run_req = pyhelayers.HeRunRequirements()
he_run_req.set_he_context_options([pyhelayers.DefaultContext()])
he_run_req.optimize_for_batch_size(16)

nn = pyhelayers.NeuralNet()
nn.encode_encrypt([os.path.join(PATH, "model.json"), os.path.join(PATH, "model.h5")], he_run_req)

The encode_encrypt method also initialized an HeContext object containing the keys. We obtain it now from the model so we can encrypt the input data.

In [ ]:
context = nn.get_created_he_context()

We will now load real samples of the MNIST dataset to classify. We will load the samples and the corresponding true labels from HDF5 files. We will also extract the first batch of samples and labels.

In [ ]:
with h5py.File(os.path.join(PATH, "x_test.h5")) as f:
    x_test = np.array(f["x_test"])
with h5py.File(os.path.join(PATH, "y_test.h5")) as f:
    y_test = np.array(f["y_test"])
    
# plain_samples, labels = utils.extract_batch(x_test, y_test, batch_size, 0)

plain_samples, labels = utils.extract_batch(x_test, y_test, 100, 0)
print('Batch of size',batch_size,'loaded')

To populate the input, we need to encode and then encrypt the values of the plain input under HE.

In [ ]:
model_io_encoder = pyhelayers.ModelIoEncoder(nn)
samples = pyhelayers.EncryptedData(context)
model_io_encoder.encode_encrypt(samples, [plain_samples])
print('Test data encrypted')

We now go ahead with the inference itself. We run the encrypted input through the encrypted NN to obtain encrypted results. This computation does not use the secret key and acts on completely encrypted values. Running the inference is done using the "predict" method of the NN, that receives the destination 3D structure to put the result of the computation in, and the input for the inference.

In [ ]:
utils.start_timer()

predictions = pyhelayers.EncryptedData(context)
nn.predict(predictions, samples)

duration=utils.end_timer('predict')
utils.report_duration('predict per sample',duration/batch_size)

In order to assess the results of the computation, we first need to decrypt them. This is done by an IO processor that has the secret key and is capable of decrypting the ciphertext and decoding it into plaintext version of the HE computation result.

In [ ]:
plain_predictions = model_io_encoder.decrypt_decode_output(predictions)
print('predictions',plain_predictions)

Now we compare the results against the expected labels and compute the confusion matrix and the accuracy.

In [ ]:
utils.assess_results(labels, plain_predictions)

<br>

References:

<sub><sup> 1.	LeCun, Yann and Cortes, Corinna. "MNIST handwritten digit database." (2010): </sup></sub>

<sub><sup> 2.	Gilad-Bachrach, R., Dowlin, N., Laine, K., Lauter, K., Naehrig, M. &amp; Wernsing, J.. (2016). CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy. Proceedings of The 33rd International Conference on Machine Learning, in Proceedings of Machine Learning Research 48:201-210 Available from https://proceedings.mlr.press/v48/gilad-bachrach16.html.
</sup></sub>
